# Experiment: 行业研究真实超时复现：半导体设备国产替代关键兑现节点

Objective:
- 真实复现行业研究输入“半导体设备国产替代，未来 12 个月最关键的兑现节点是什么”后的失败链路。
- 先直接探测 Python intelligence service，再通过真实 `QuickResearchContractLangGraph(v3)` 运行行业研究，确认是否落到 `INTELLIGENCE_DATA_UNAVAILABLE` 超时。
- 输出当前配置的 timeout、HTTP 预检结果、workflow 失败节点、run 失败信息与事件流，便于直接定位问题。 


## Setup And Reproducibility

- notebook 会从 `.env`、`.env.local`、`deploy/.env` 尝试补齐环境变量，然后调用同目录的 `semiconductor-equipment-localization-e2e.ts` harness。
- harness 使用真实的 `PythonIntelligenceDataClient`、`ConfidenceAnalysisService`、`QuickResearchWorkflowService`、`QuickResearchContractLangGraph` 和 `WorkflowExecutionService`；DeepSeek 合约生成默认关闭，只为避免 LLM 把复现带偏到别的错误。
- 只把数据库持久化与 Redis runtime store 换成内存 harness，避免为了复现上游超时还要求你先准备测试库。 


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path



def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError(f'Unable to locate repo root from {start}')


def load_env_file(path: Path) -> dict[str, str]:
    loaded: dict[str, str] = {}
    if not path.exists():
        return loaded

    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if os.getenv(key) is None:
            os.environ[key] = value
            loaded[key] = '<set>' if key.endswith('_KEY') or 'PASSWORD' in key or 'SECRET' in key else value
    return loaded


REPO_ROOT = find_repo_root(Path.cwd().resolve())
QUESTION = '半导体设备国产替代，未来 12 个月最关键的兑现节点是什么'
HARNESS_PATH = REPO_ROOT / 'tests' / 'jupyter_notebooks' / 'semiconductor-equipment-localization-e2e.ts'
NPM_EXECUTABLE = 'npm.cmd' if os.name == 'nt' else 'npm'

loaded_env = {}
for env_path in [REPO_ROOT / '.env', REPO_ROOT / '.env.local', REPO_ROOT / 'deploy' / '.env']:
    loaded_env[str(env_path.relative_to(REPO_ROOT))] = load_env_file(env_path)

CONFIGURED_SERVICE_URL = os.getenv('PYTHON_INTELLIGENCE_SERVICE_URL')
BASE_URL = os.getenv('WORKFLOW_REPRO_BASE_URL', 'http://127.0.0.1:8000').rstrip('/')
WORKFLOW_TIMEOUT_MS = int(os.getenv('WORKFLOW_REPRO_TIMEOUT_MS', os.getenv('PYTHON_INTELLIGENCE_SERVICE_TIMEOUT_MS', '300000')))
PRECHECK_TIMEOUT_MS = int(os.getenv('WORKFLOW_PRECHECK_TIMEOUT_MS', '5000'))
DISABLE_DEEPSEEK = os.getenv('WORKFLOW_REPRO_DISABLE_DEEPSEEK', '1') != '0'

{
    'repo_root': str(REPO_ROOT),
    'question': QUESTION,
    'harness_path': str(HARNESS_PATH),
    'harness_exists': HARNESS_PATH.exists(),
    'base_url': BASE_URL,
    'configured_service_url': CONFIGURED_SERVICE_URL,
    'workflow_timeout_ms': WORKFLOW_TIMEOUT_MS,
    'precheck_timeout_ms': PRECHECK_TIMEOUT_MS,
    'npm_executable': NPM_EXECUTABLE,
    'disable_deepseek': DISABLE_DEEPSEEK,
    'loaded_env_files': loaded_env,
}


## Preflight

- 真实 preflight 由 TS harness 在同一个 Node 运行时里执行，避免 notebook 先额外打一轮 HTTP，把现场状态搅乱。
- 你在下面看到的 `preflight` 结果，和真正触发 workflow 的那次进程是同一份数据。 


In [ ]:
{
    'base_url': BASE_URL,
    'workflow_timeout_ms': WORKFLOW_TIMEOUT_MS,
    'precheck_timeout_ms': PRECHECK_TIMEOUT_MS,
    'disable_deepseek': DISABLE_DEEPSEEK,
}


## Real Workflow Reproduction

- 这一步运行真实 quick research v3 图，使用当前配置的 `PYTHON_INTELLIGENCE_SERVICE_TIMEOUT_MS`。
- 默认会把 `WORKFLOW_REPRO_DISABLE_DEEPSEEK=1` 传给 harness，让 contract/brief 走代码 fallback，避免外部 LLM 把复现带偏到 `clarification_required` 或 `INTELLIGENCE_LLM_PARSE_FAILED`。
- 如果你要严格复现线上那条 `300000ms` 报错，不要改 `WORKFLOW_REPRO_TIMEOUT_MS`；如果只是快速确认链路，也可以在启动 notebook 前把它临时设小。 


In [ ]:
completed = subprocess.run(
    [NPM_EXECUTABLE, 'exec', 'tsx', '--', str(HARNESS_PATH)],
    cwd=REPO_ROOT,
    env={
        **os.environ,
        'SKIP_ENV_VALIDATION': '1',
        'WORKFLOW_REPRO_BASE_URL': BASE_URL,
        'WORKFLOW_REPRO_TIMEOUT_MS': str(WORKFLOW_TIMEOUT_MS),
        'WORKFLOW_PRECHECK_TIMEOUT_MS': str(PRECHECK_TIMEOUT_MS),
        'WORKFLOW_REPRO_DISABLE_DEEPSEEK': '1' if DISABLE_DEEPSEEK else '0',
    },
    capture_output=True,
    text=True,
    encoding='utf-8',
    errors='replace',
    check=True,
)

if completed.stderr.strip():
    print(completed.stderr)

harness_result = json.loads(completed.stdout)
harness_result['workflow']


## Results

- 这里的断言默认就是为了复现当前问题，所以它要求 workflow 最终失败，并且错误码是 `INTELLIGENCE_DATA_UNAVAILABLE`。
- 如果某天这里不再失败了，通常意味着问题已经被修掉，或者当前 notebook 指向的 base URL / timeout 跟你复现问题时的环境不一致。 


In [ ]:
workflow = harness_result['workflow']
expected_timeout_message = f"Intelligence 数据服务请求超时 ({WORKFLOW_TIMEOUT_MS}ms)"

assert workflow['picked'] is True
assert workflow['startedStatus'] == 'PENDING'
assert workflow['finalStatus'] == 'FAILED'
assert workflow['runFailure']['errorCode'] == 'INTELLIGENCE_DATA_UNAVAILABLE'
assert expected_timeout_message in workflow['runFailure']['errorMessage']
assert workflow['reproducedTimeout'] is True

summary = {
    'config': harness_result['config'],
    'preflight': harness_result['preflight'],
    'workflow_failure': workflow['runFailure'],
    'failed_node_record': workflow['failedNode'],
    'current_node_at_failure': workflow['currentNodeKey'],
    'progress_percent': workflow['progressPercent'],
    'node_runs': workflow['nodeRuns'],
    'published_event_types': workflow['publishedEventTypes'],
    'checkpoint_cleared': workflow['checkpointCleared'],
}

summary


## Notes

- 如果 `failed_node_record.nodeKey` 和 `current_node_at_failure` 不一致，说明 run 失败归因可能存在偏移；这是这次真实复现里额外暴露出来的一个现象。
- notebook 本身不修这个问题，只负责把真实超时链路和现场状态稳定地采出来。 
